In [1]:
import ray
from ray.util.dask import enable_dask_on_ray
import dask.dataframe as dd
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
import pandas as pd
from PIL import Image
import time
from IPython.display import display

if ray.is_initialized():
    ray.shutdown()

runtime_env = {
    "pip": [
        "datashader", 
        "pytz", 
        "pandas", 
        "pyarrow", 
        "colorcet", 
        "dask[complete]"
    ]
}

ray.init(address="auto", runtime_env=runtime_env, allow_multiple=True)
enable_dask_on_ray()

print("Cluster ready and synced.")

2026-03-26 12:47:16,366	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-03-26 12:47:16,385	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Cluster ready and synced.


/opt/conda/lib/python3.11/site-packages/dask/config.py:786: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


In [3]:
print(ray.cluster_resources())

Exception: Ray Client is not connected. Please connect by calling `ray.init`.

In [18]:
# Configuration
Image.MAX_IMAGE_PIXELS = None 
W, H = 6000, 3000

# Input
t = input("Time (YYYY or YYYY-MM) [2025-01]: ") or "2025-01"
sp = input("Species [*]: ") or "*"
sub = input("Sub-Species [*]: ") or "*"
bb = [float(n.strip()) for n in (input("BBox [Global]: ") or "-180,180,-90,90").split(",")]

start = time.time()
y = t.split("-")[0]

# Construct paths based on your folder structure
g_path = f"/mnt/shared_data/finflow/gfw_raw/{y}/{t if '-' in t else '*'}.parquet"
o_path = f"/mnt/shared_data/finflow/obis_raw/{sp}/{sub}/*.parquet"

# Distributed Loading
g_ddf = dd.read_parquet(g_path, columns=['lon', 'lat', 'hours'])
o_ddf = dd.read_parquet(o_path, columns=['decimalLongitude', 'decimalLatitude', 'eventDate']).rename(columns={'decimalLongitude': 'lon', 'decimalLatitude': 'lat'})
o_filtered = o_ddf[(o_ddf.lon.between(bb[0], bb[1])) & (o_ddf.lat.between(bb[2], bb[3]))]

# Parallel Rendering
cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(bb[0], bb[1]), y_range=(bb[2], bb[3]))
agg_g = cvs.points(g_ddf, 'lon', 'lat', ds.sum('hours'))
agg_o = cvs.points(o_filtered, 'lon', 'lat', ds.count())

# Composition
img = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
img.alpha_composite(tf.shade(agg_g, cmap=cc.fire, how='log').to_pil().convert("RGBA"))

if not o_filtered.head(1).empty:
    img.alpha_composite(tf.shade(agg_o, cmap=["#90ee90", "#00ff00"], how='log').to_pil().convert("RGBA"))

print(f"Total processing time: {time.time()-start:.2f}s")
display(img)

Time (YYYY or YYYY-MM) [2025-01]:  
Species [*]:  
Sub-Species [*]:  
BBox [Global]:  


2026-03-26 12:42:20,666	WARNING worker.py:1659 -- SIGTERM handler is not set because current thread is not the main thread.
2026-03-26 12:42:20,667	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-03-26 12:42:20,669	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver


ValueError: The client has already connected to the cluster with allow_multiple=True. Please set allow_multiple=True to proceed

In [19]:
print(ray.cluster_resources())

Exception: Ray Client is not connected. Please connect by calling `ray.init`.